# ALM Cash Flow Projections from Random Survival Forest

This notebook converts the cause-specific RSF model outputs (notebook 07) into projected mortgage cash flows for Asset-Liability Management (ALM) analysis.

## Key Differences from Cox-based ALM (Notebook 14)

| Aspect | Cox (Notebook 14) | RSF (This Notebook) |
|--------|-------------------|---------------------|
| **Hazard model** | h(t\|X) = h0(t) exp(X*beta) | Non-parametric survival curves S(t\|X) |
| **Features** | Time-varying covariates (3D matrix) | Snapshot features (2D matrix) |
| **Hazard extraction** | Baseline hazard + coefficients | h(t) = 1 - S(t)/S(t-1) |
| **Scenario analysis** | Month-by-month covariate paths | Modified feature vectors |
| **Flexibility** | Proportional hazards assumption | Captures non-linear effects |

## Overview
1. **Setup**: Load fitted RSF models and prepare loan data
2. **RSF survival curves**: Extract and visualize cause-specific survival and hazards
3. **Backtest: Predicted vs Realized (Fold 10)**: Out-of-sample validation
4. **Single loan walkthrough**: Full algorithm on one loan
5. **Portfolio projection**: Aggregate monthly cash flows under base scenario
6. **Scenario analysis**: Rate shocks, HPI stress, recession scenarios
7. **Risk metrics comparison**: NPV, duration, convexity across scenarios
8. **Portfolio segmentation**: Risk metrics by vintage, FICO, LTV

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

from src.alm.rsf_cash_flow_engine import RSFCashFlowConfig, RSFCashFlowEngine
from src.alm.risk_metrics import (
    compute_npv, compute_modified_duration, compute_modified_convexity,
    compute_wal, compute_all_risk_metrics,
)

sns.set_style('whitegrid')
%matplotlib inline

DATA_DIR = Path('../data/processed')
EXTERNAL_DIR = Path('../data/external')
MODELS_DIR = Path('../models')
FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Imports complete.')

Imports complete.


---

## 1. Setup: Load RSF Models and Data

In [2]:
# Load fitted RSF models (from notebook 07)
with open(MODELS_DIR / 'rsf_prepay.pkl', 'rb') as f:
    rsf_prepay = pickle.load(f)
with open(MODELS_DIR / 'rsf_default.pkl', 'rb') as f:
    rsf_default = pickle.load(f)

print(f'RSF Prepayment model: {rsf_prepay.n_estimators} trees, max_depth={rsf_prepay.max_depth}')
print(f'RSF Default model: {rsf_default.n_estimators} trees, max_depth={rsf_default.max_depth}')
print(f'Unique event times (prepay): {len(rsf_prepay.unique_times_)} (max={rsf_prepay.unique_times_.max():.0f})')
print(f'Unique event times (default): {len(rsf_default.unique_times_)} (max={rsf_default.unique_times_.max():.0f})')

RSF Prepayment model: 100 trees, max_depth=10
RSF Default model: 100 trees, max_depth=10
Unique event times (prepay): 184 (max=184)
Unique event times (default): 184 (max=184)


In [3]:
# Feature columns (must match notebook 07 exactly)
# Note: RSF used log_upb instead of orig_upb and bal_repaid_lag1 instead of bal_repaid
STATIC_FEATURES = ['int_rate', 'log_upb', 'fico_score', 'dti_r', 'ltv_r']
BEHAVIORAL_FEATURES = ['bal_repaid_lag1', 't_act_12m', 't_del_30d_12m', 't_del_60d_12m']
MACRO_FEATURES = [
    'hpi_st_d_t_o', 'ppi_c_FRMA', 'TB10Y_d_t_o', 'FRMA30Y_d_t_o',
    'ppi_o_FRMA', 'hpi_st_log12m', 'hpi_r_st_us', 'st_unemp_r12m',
    'st_unemp_r3m', 'TB10Y_r12m', 'T10Y3MM', 'T10Y3MM_r12m',
]

FEATURE_COLS = STATIC_FEATURES + BEHAVIORAL_FEATURES + MACRO_FEATURES
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

Features (21): ['int_rate', 'log_upb', 'fico_score', 'dti_r', 'ltv_r', 'bal_repaid_lag1', 't_act_12m', 't_del_30d_12m', 't_del_60d_12m', 'hpi_st_d_t_o', 'ppi_c_FRMA', 'TB10Y_d_t_o', 'FRMA30Y_d_t_o', 'ppi_o_FRMA', 'hpi_st_log12m', 'hpi_r_st_us', 'st_unemp_r12m', 'st_unemp_r3m', 'TB10Y_r12m', 'T10Y3MM', 'T10Y3MM_r12m']


In [ ]:
# Load panel data
panel_df = pd.read_parquet(DATA_DIR / 'loan_month_panel.parquet')
print(f'Panel: {len(panel_df):,} loan-months, {panel_df["loan_sequence_number"].nunique():,} loans')

# Load survival data for orig_loan_term and origination-time macro values
surv_df = pd.read_parquet(DATA_DIR / 'survival_data_blumenstock.parquet')

# Sort chronologically
panel_df = panel_df.sort_values(['loan_sequence_number', 'loan_age'])

# === Origination-time features (first observation per loan) ===
# This avoids data leakage: features are measured BEFORE any outcome is known.
origin_df = panel_df.groupby('loan_sequence_number').first().reset_index()

# === Terminal outcomes (last observation per loan) ===
terminal_df = panel_df.groupby('loan_sequence_number').last().reset_index()

# Join terminal outcomes (event_code, loan_age at event) onto origin_df
origin_df = origin_df.drop(columns=['event_code'], errors='ignore')
origin_df = origin_df.merge(
    terminal_df[['loan_sequence_number', 'event_code', 'loan_age']].rename(
        columns={'loan_age': 'terminal_loan_age'}
    ),
    on='loan_sequence_number',
    how='left',
)

# bal_repaid at origination is ~0 by definition; use it directly
origin_df['bal_repaid_lag1'] = origin_df.get('bal_repaid', 0.0)

# Log transform UPB
origin_df['log_upb'] = np.log(origin_df['orig_upb'])

# Join orig_loan_term and origination-time macro from survival data
orig_cols = ['loan_sequence_number', 'orig_loan_term', 'orig_MORTGAGE30US', 'orig_DGS10', 'orig_state_hpi']
orig_info = surv_df[orig_cols].drop_duplicates(subset=['loan_sequence_number'])
origin_df = origin_df.merge(orig_info, on='loan_sequence_number', how='left')
origin_df['orig_loan_term'] = origin_df['orig_loan_term'].fillna(360).astype(int)

# current_loan_age = loan_age at first observation (start of projection)
origin_df['current_loan_age'] = origin_df['loan_age'].astype(int)

# Drop rows with missing features
origin_df = origin_df.dropna(subset=FEATURE_COLS).copy()

print(f'Origination observations: {len(origin_df):,} loans')
print(f'First obs loan_age: mean={origin_df["current_loan_age"].mean():.1f}, '
      f'median={origin_df["current_loan_age"].median():.0f}')
print(f'\nEvent distribution (terminal outcomes):')
print(origin_df['event_code'].value_counts().rename({0: 'Censored', 1: 'Prepay', 2: 'Default'}))

In [5]:
# Load macro data
macro_df = pd.read_parquet(EXTERNAL_DIR / 'fred_monthly_panel.parquet')
state_hpi_df = pd.read_parquet(EXTERNAL_DIR / 'state_hpi.parquet')
state_unemp_df = pd.read_parquet(EXTERNAL_DIR / 'state_unemployment.parquet')

print(f'Macro data: {len(macro_df)} months, last: {macro_df.index[-1].strftime("%Y-%m")}')
print(f'State HPI: {state_hpi_df.shape[1]} states, last: {state_hpi_df.index[-1].strftime("%Y-%m")}')
print(f'State unemp: {state_unemp_df.shape[1]} states, last: {state_unemp_df.index[-1].strftime("%Y-%m")}')
print(f'\nLast observed rates:')
print(f'  MORTGAGE30US: {macro_df["MORTGAGE30US"].iloc[-1]:.2f}%')
print(f'  DGS10: {macro_df["DGS10"].iloc[-1]:.2f}%')
print(f'  DGS3MO: {macro_df["DGS3MO"].iloc[-1]:.2f}%')

Macro data: 337 months, last: 2026-01
State HPI: 51 states, last: 2025-07
State unemp: 51 states, last: 2025-11

Last observed rates:
  MORTGAGE30US: 6.11%
  DGS10: 4.17%
  DGS3MO: 3.65%


---

## 2. RSF Survival Curves and Implied Hazards

Unlike Cox models where baseline hazards are stored directly, RSF hazards are
extracted from the predicted survival functions:

$$h_k(t \mid X) = 1 - \frac{S_k(t \mid X)}{S_k(t-1 \mid X)}$$

These hazards are individual-specific (no proportional hazards assumption).

In [ ]:
# Predict survival curves for a sample of loans to visualize
np.random.seed(42)
sample_idx = np.random.choice(len(origin_df), size=min(500, len(origin_df)), replace=False)
sample_df = origin_df.iloc[sample_idx]
X_sample = sample_df[FEATURE_COLS].values

# Create RSF engine
config = RSFCashFlowConfig(lgd=0.25)
engine = RSFCashFlowEngine(rsf_prepay, rsf_default, config=config)

# Predict survival curves on monthly grid
max_month = 200
S_prepay, S_default = engine.predict_survival_curves(X_sample, max_month=max_month)

print(f'Survival curves: prepay={S_prepay.shape}, default={S_default.shape}')
effective_max = S_prepay.shape[1] - 1
print(f'Effective max month (clamped to model domain): {effective_max}')
if effective_max >= 120:
    print(f'\nPrepay S(120m): mean={S_prepay[:, 120].mean():.4f}, std={S_prepay[:, 120].std():.4f}')
    print(f'Default S(120m): mean={S_default[:, 120].mean():.4f}, std={S_default[:, 120].std():.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
months = np.arange(S_prepay.shape[1])

# --- Prepay survival ---
ax = axes[0, 0]
for i in range(min(50, len(S_prepay))):
    ax.plot(months, S_prepay[i], color='steelblue', alpha=0.1, lw=0.5)
ax.plot(months, S_prepay.mean(axis=0), 'k-', lw=2, label='Mean S(t)')
ax.fill_between(months,
    np.percentile(S_prepay, 25, axis=0),
    np.percentile(S_prepay, 75, axis=0),
    alpha=0.2, color='steelblue', label='IQR')
ax.set_title('Prepay Survival Curves')
ax.set_xlabel('Month')
ax.set_ylabel('S(t)')
ax.legend()

# --- Default survival ---
ax = axes[0, 1]
for i in range(min(50, len(S_default))):
    ax.plot(months, S_default[i], color='indianred', alpha=0.1, lw=0.5)
ax.plot(months, S_default.mean(axis=0), 'k-', lw=2, label='Mean S(t)')
ax.fill_between(months,
    np.percentile(S_default, 25, axis=0),
    np.percentile(S_default, 75, axis=0),
    alpha=0.2, color='indianred', label='IQR')
ax.set_title('Default Survival Curves')
ax.set_xlabel('Month')
ax.set_ylabel('S(t)')
ax.legend()

# --- Prepay hazard ---
ax = axes[1, 0]
h_p = 1 - S_prepay[:, 1:] / np.maximum(S_prepay[:, :-1], 1e-10)
h_p = np.maximum(h_p, 0)
ax.plot(months[1:], h_p.mean(axis=0), 'steelblue', lw=2)
ax.fill_between(months[1:],
    np.percentile(h_p, 25, axis=0),
    np.percentile(h_p, 75, axis=0),
    alpha=0.2, color='steelblue')
ax.set_title('Prepay Hazard Rate')
ax.set_xlabel('Month')
ax.set_ylabel('h(t)')

# --- Default hazard ---
ax = axes[1, 1]
h_d = 1 - S_default[:, 1:] / np.maximum(S_default[:, :-1], 1e-10)
h_d = np.maximum(h_d, 0)
ax.plot(months[1:], h_d.mean(axis=0), 'indianred', lw=2)
ax.fill_between(months[1:],
    np.percentile(h_d, 25, axis=0),
    np.percentile(h_d, 75, axis=0),
    alpha=0.2, color='indianred')
ax.set_title('Default Hazard Rate')
ax.set_xlabel('Month')
ax.set_ylabel('h(t)')

plt.suptitle('RSF Survival Curves and Hazard Rates', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Backtest: Predicted vs Realized (Fold 10)

The RSF models were trained on folds 0-9. Fold 10 was held out entirely.
We use each test loan's **origination-time features** (first observation) to predict its
survival/CIF curves, then compare against realized outcomes (terminal event and loan age).

Using origination features avoids data leakage: terminal features contain information
that is only available *after* the outcome is known (e.g., high delinquency before default,
bal_repaid near 100% before prepayment).

In [ ]:
# === Prepare fold-10 test data (origination features) ===
LGD = 0.25

test_df = origin_df[origin_df['fold'] == 10].copy()
test_df = test_df.dropna(subset=FEATURE_COLS).copy()

X_test = test_df[FEATURE_COLS].values
n_test = len(test_df)

print(f'Test set (fold 10): {n_test:,} loans')
print(f'  Prepayments: {(test_df["event_code"] == 1).sum():,}')
print(f'  Defaults: {(test_df["event_code"] == 2).sum():,}')
print(f'  Censored: {(test_df["event_code"] == 0).sum():,}')
print(f'  Mean origination loan_age: {test_df["current_loan_age"].mean():.1f}')

# Predict survival curves for test loans
max_test_age = int(test_df['terminal_loan_age'].max()) + 10
S_test_prepay, S_test_default = engine.predict_survival_curves(X_test, max_month=max_test_age)

print(f'\nSurvival curves shape: {S_test_prepay.shape}')

In [ ]:
# === Compute predicted CIF and compare with realized ===

# Predicted cause-specific hazards on monthly grid
h_test_prepay = np.maximum(0, 1.0 - S_test_prepay[:, 1:] / np.maximum(S_test_prepay[:, :-1], 1e-10))
h_test_default = np.maximum(0, 1.0 - S_test_default[:, 1:] / np.maximum(S_test_default[:, :-1], 1e-10))

# Combined survival under competing risks
T_test = h_test_prepay.shape[1]
survival_combined = np.zeros((n_test, T_test))
f_prepay_test = np.zeros((n_test, T_test))
f_default_test = np.zeros((n_test, T_test))

s_prev = np.ones(n_test)
for t in range(T_test):
    h_total = np.minimum(h_test_prepay[:, t] + h_test_default[:, t], 0.999)
    scale = np.where(
        (h_test_prepay[:, t] + h_test_default[:, t]) > 0.999,
        0.999 / (h_test_prepay[:, t] + h_test_default[:, t]),
        1.0,
    )
    hp = h_test_prepay[:, t] * scale
    hd = h_test_default[:, t] * scale
    
    f_prepay_test[:, t] = hp * s_prev
    f_default_test[:, t] = hd * s_prev
    s_prev = s_prev * (1.0 - hp - hd)
    survival_combined[:, t] = s_prev

# Predicted CIF (loan-level, then average across loans)
pred_cif_prepay = np.cumsum(f_prepay_test, axis=1).mean(axis=0)
pred_cif_default = np.cumsum(f_default_test, axis=1).mean(axis=0)

# Realized CIF: aggregate by terminal_loan_age (the age at which the event occurred)
loan_ages = test_df['terminal_loan_age'].values.astype(int)
event_codes = test_df['event_code'].values

max_age_obs = loan_ages.max()
real_prepay_count = np.zeros(max_age_obs + 1)
real_default_count = np.zeros(max_age_obs + 1)

for i in range(n_test):
    age = loan_ages[i]
    if event_codes[i] == 1:
        real_prepay_count[age] += 1
    elif event_codes[i] == 2:
        real_default_count[age] += 1

real_cif_prepay = np.cumsum(real_prepay_count) / n_test
real_cif_default = np.cumsum(real_default_count) / n_test

print(f'Max observed terminal age: {max_age_obs}')
print(f'\nAt age {min(120, max_age_obs)}:')
t_check = min(120, max_age_obs)
print(f'  Predicted CIF prepay: {pred_cif_prepay[t_check-1]:.4f}')
print(f'  Realized CIF prepay:  {real_cif_prepay[t_check]:.4f}')
print(f'  Predicted CIF default: {pred_cif_default[t_check-1]:.4f}')
print(f'  Realized CIF default:  {real_cif_default[t_check]:.4f}')

In [ ]:
# === Predicted and realized cash flows (fold 10) ===

# Amortization for each test loan at its terminal event age
int_rate = test_df['int_rate'].values
orig_upb = test_df['orig_upb'].values
term = test_df['orig_loan_term'].values.astype(float)
loan_age = test_df['terminal_loan_age'].values.astype(float)

monthly_rate = int_rate / 100.0 / 12.0
payment = np.where(
    monthly_rate > 0,
    orig_upb * monthly_rate / (1.0 - (1.0 + monthly_rate) ** (-term)),
    orig_upb / term,
)

# UPB at each loan's terminal age
factor = (1.0 + monthly_rate) ** loan_age
upb_at_event = np.where(
    monthly_rate > 0,
    orig_upb * factor - payment * (factor - 1.0) / monthly_rate,
    orig_upb - payment * loan_age,
)
upb_at_event = np.maximum(upb_at_event, 0.0)

interest_at_event = upb_at_event * monthly_rate

# Predicted: probability-weighted by survival at terminal event time
pred_cf_components = {
    'pred_interest': np.zeros(n_test),
    'pred_prepay': np.zeros(n_test),
    'pred_recovery': np.zeros(n_test),
    'pred_loss': np.zeros(n_test),
}

for i in range(n_test):
    age = int(loan_age[i])
    if age <= 0 or age > T_test:
        continue
    s_prev_i = survival_combined[i, age - 2] if age > 1 else 1.0
    pred_cf_components['pred_interest'][i] = s_prev_i * interest_at_event[i]
    pred_cf_components['pred_prepay'][i] = f_prepay_test[i, age - 1] * upb_at_event[i]
    pred_cf_components['pred_recovery'][i] = f_default_test[i, age - 1] * upb_at_event[i] * (1 - LGD)
    pred_cf_components['pred_loss'][i] = f_default_test[i, age - 1] * upb_at_event[i] * LGD

# Realized
is_prepay = (event_codes == 1)
is_default = (event_codes == 2)

real_cf_components = {
    'real_interest': np.where(~is_default, interest_at_event, 0.0),
    'real_prepay': np.where(is_prepay, upb_at_event, 0.0),
    'real_recovery': np.where(is_default, upb_at_event * (1 - LGD), 0.0),
    'real_loss': np.where(is_default, upb_at_event * LGD, 0.0),
}

print('=== Cash Flow Totals (Fold 10) ===')
for component in ['interest', 'prepay', 'recovery', 'loss']:
    pred = pred_cf_components[f'pred_{component}'].sum()
    real = real_cf_components[f'real_{component}'].sum()
    ratio = pred / real if real != 0 else float('nan')
    print(f'  {component:>10s}:  predicted ${pred/1e6:>8.2f}M   realized ${real/1e6:>8.2f}M   ratio {ratio:.3f}')

In [ ]:
# === Backtest plots ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Common x-axis limit
x_max = min(max_age_obs, T_test)
ages_plot = np.arange(1, x_max + 1)

# 1. CIF: Prepayment
ax = axes[0, 0]
ax.plot(ages_plot, real_cif_prepay[1:x_max+1], 'k-', lw=2, label='Realized')
ax.plot(ages_plot, pred_cif_prepay[:x_max], '--', color='steelblue', lw=2, label='RSF Predicted')
ax.set_title('Cumulative Incidence: Prepayment')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('CIF')
ax.legend()

# 2. CIF: Default
ax = axes[0, 1]
ax.plot(ages_plot, real_cif_default[1:x_max+1], 'k-', lw=2, label='Realized')
ax.plot(ages_plot, pred_cif_default[:x_max], '--', color='indianred', lw=2, label='RSF Predicted')
ax.set_title('Cumulative Incidence: Default')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('CIF')
ax.legend()

# 3. Average hazard: Prepayment (smoothed)
ax = axes[1, 0]
window = 6
h_pred_avg = pd.Series(h_test_prepay[:, :x_max].mean(axis=0)).rolling(window, center=True, min_periods=1).mean()
# Realized hazard per period
real_hazard_prepay = real_prepay_count[1:x_max+1] / np.maximum(
    n_test - np.cumsum(real_prepay_count + real_default_count)[0:x_max], 1
)
h_real_avg = pd.Series(real_hazard_prepay).rolling(window, center=True, min_periods=1).mean()
ax.plot(ages_plot, h_real_avg.values, 'k-', lw=1.5, label=f'Realized ({window}m avg)')
ax.plot(ages_plot, h_pred_avg.values, '--', color='steelblue', lw=1.5, label=f'RSF Predicted ({window}m avg)')
ax.set_title('Prepayment Hazard Rate')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('h(t)')
ax.legend()

# 4. Average survival (combined competing risks)
ax = axes[1, 1]
avg_surv = survival_combined[:, :x_max].mean(axis=0)
# Realized survival: Kaplan-Meier style
total_events = np.cumsum(real_prepay_count + real_default_count)
real_surv = 1.0 - total_events[1:x_max+1] / n_test
ax.plot(ages_plot, real_surv, 'k-', lw=2, label='Realized')
ax.plot(ages_plot, avg_surv, '--', color='steelblue', lw=2, label='RSF Predicted')
ax.set_title('Overall Survival (Competing Risks)')
ax.set_xlabel('Loan Age (months)')
ax.set_ylabel('S(t)')
ax.legend()

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle('Backtest: RSF Predicted vs Realized (Fold 10, Out-of-Sample)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_rsf_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Backtest summary statistics ===
print('=== Backtest Summary: RSF Predicted vs Realized (Fold 10) ===\n')

horizons = [24, 48, 72, 120, 150]
print(f'{"Horizon":>8s}  {"Pred CIF_P":>10s}  {"Real CIF_P":>10s}  {"Pred CIF_D":>10s}  {"Real CIF_D":>10s}')
print('-' * 60)
for h in horizons:
    if h > x_max:
        continue
    print(f'{h:>6d}m  {pred_cif_prepay[h-1]:>10.4f}  {real_cif_prepay[h]:>10.4f}  '
          f'{pred_cif_default[h-1]:>10.4f}  {real_cif_default[h]:>10.4f}')

# Accounting identity check (average across loans at last observed time)
t_end = min(x_max, T_test) - 1
avg_s = survival_combined[:, t_end].mean()
avg_cif_p = np.cumsum(f_prepay_test, axis=1)[:, t_end].mean()
avg_cif_d = np.cumsum(f_default_test, axis=1)[:, t_end].mean()
print(f'\n=== Accounting Identity at month {t_end+1} (average across loans) ===')
print(f'  S(t):         {avg_s:.6f}')
print(f'  CIF_prepay:   {avg_cif_p:.6f}')
print(f'  CIF_default:  {avg_cif_d:.6f}')
print(f'  Sum:          {avg_s + avg_cif_p + avg_cif_d:.6f} (should be ~1.0)')

---

## 4. Single Loan Walkthrough

In [ ]:
# Pick a representative loan: 30-year, still active (censored)
active_loans = origin_df[
    (origin_df['orig_loan_term'] == 360) &
    (origin_df['event_code'] == 0) &
    (origin_df['current_loan_age'] > 12) &
    (origin_df['orig_MORTGAGE30US'].notna())
].copy()
example_loan = active_loans.iloc[0:1].copy()

print('=== Example Loan ===')
for col in ['loan_sequence_number', 'int_rate', 'orig_upb', 'fico_score', 'dti_r', 'ltv_r',
            'orig_loan_term', 'current_loan_age', 'property_state']:
    if col in example_loan.columns:
        print(f'  {col}: {example_loan[col].iloc[0]}')

In [ ]:
# Project cash flows for single loan using RSF engine
X_single = example_loan[FEATURE_COLS].values

cf_single = engine.project_cash_flows(example_loan, X_single)

# Verify accounting identity
T = cf_single['survival'].shape[1]
cif_prepay = np.cumsum(cf_single['f_prepay'][0])
cif_default = np.cumsum(cf_single['f_default'][0])
identity_check = cf_single['survival'][0, -1] + cif_prepay[-1] + cif_default[-1]

print(f'=== Single Loan Cash Flow Summary (RSF) ===')
print(f'Projection horizon: {T} months')
print(f'Total interest: ${cf_single["interest"][0].sum():,.2f}')
print(f'Total sched principal: ${cf_single["scheduled_principal"][0].sum():,.2f}')
print(f'Total prepayment: ${cf_single["prepayment"][0].sum():,.2f}')
print(f'Total recovery: ${cf_single["recovery"][0].sum():,.2f}')
print(f'Total loss: ${cf_single["loss"][0].sum():,.2f}')
print(f'Total CF: ${cf_single["total_cf"][0].sum():,.2f}')
print(f'\n=== Accounting Identity Check ===')
print(f'S({T}): {cf_single["survival"][0, -1]:.6f}')
print(f'CIF_prepay({T}): {cif_prepay[-1]:.6f}')
print(f'CIF_default({T}): {cif_default[-1]:.6f}')
print(f'Sum: {identity_check:.6f} (should be ~1.0)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
months = np.arange(1, T + 1)

# Survival and CIF
ax = axes[0, 0]
ax.plot(months, cf_single['survival'][0], 'k-', lw=2, label='Survival S(t)')
ax.plot(months, cif_prepay, '--', color='steelblue', lw=1.5, label='CIF prepay')
ax.plot(months, cif_default, '--', color='indianred', lw=1.5, label='CIF default')
ax.fill_between(months, 0, cif_default, alpha=0.2, color='indianred')
ax.fill_between(months, cif_default, cif_default + cif_prepay, alpha=0.2, color='steelblue')
ax.set_title('Survival and Cumulative Incidence')
ax.set_xlabel('Month')
ax.set_ylabel('Probability')
ax.legend()

# Expected cash flows
ax = axes[0, 1]
ax.stackplot(months,
    cf_single['interest'][0],
    cf_single['scheduled_principal'][0],
    cf_single['prepayment'][0],
    cf_single['recovery'][0],
    labels=['Interest', 'Sched Principal', 'Prepayment', 'Recovery'],
    colors=['#4e79a7', '#59a14f', '#f28e2b', '#e15759'],
    alpha=0.8)
ax.set_title('Expected Monthly Cash Flows')
ax.set_xlabel('Month')
ax.set_ylabel('$')
ax.legend(loc='upper right', fontsize=8)

# Hazard rates
ax = axes[1, 0]
s_start = np.ones(T)
s_start[1:] = cf_single['survival'][0, :-1]
h_prepay_realized = np.where(s_start > 1e-10, cf_single['f_prepay'][0] / s_start, 0)
h_default_realized = np.where(s_start > 1e-10, cf_single['f_default'][0] / s_start, 0)
ax.plot(months, h_prepay_realized, color='steelblue', lw=1.5, label='h_prepay(t)')
ax.plot(months, h_default_realized, color='indianred', lw=1.5, label='h_default(t)')
ax.set_title('Cause-Specific Hazard Rates')
ax.set_xlabel('Month')
ax.set_ylabel('Hazard h(t)')
ax.legend()

# Amortization
ax = axes[1, 1]
ax.plot(months, cf_single['upb_schedule'][0], 'k-', lw=1.5, label='Scheduled UPB')
ax.set_title('Amortization Schedule')
ax.set_xlabel('Month')
ax.set_ylabel('UPB ($)')
ax.legend()

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Single Loan Walkthrough (RSF): {example_loan["loan_sequence_number"].iloc[0]}', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_rsf_single_loan_walkthrough.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 5. Portfolio Projection (Base Scenario)

In [ ]:
# Prepare portfolio: active (censored) loans with complete data
# Only censored loans still have future cash flows to project
portfolio = origin_df[
    (origin_df['event_code'] == 0) &
    origin_df['orig_MORTGAGE30US'].notna() &
    origin_df['orig_DGS10'].notna() &
    origin_df['orig_state_hpi'].notna()
].copy()

# Only loans with remaining term
portfolio['remaining_term'] = portfolio['orig_loan_term'] - portfolio['current_loan_age']
portfolio = portfolio[portfolio['remaining_term'] > 0].copy()

print(f'Portfolio: {len(portfolio):,} active loans')
print(f'Total orig UPB: ${portfolio["orig_upb"].sum():,.0f}')
print(f'Average remaining term: {portfolio["remaining_term"].mean():.0f} months')
print(f'\nBy orig_loan_term:')
print(portfolio.groupby('orig_loan_term').agg(
    n_loans=('loan_sequence_number', 'count'),
    avg_rate=('int_rate', 'mean'),
    avg_fico=('fico_score', 'mean'),
    avg_age=('current_loan_age', 'mean'),
).round(1))

In [ ]:
def build_rsf_features(
    loans_df, macro_df, state_hpi_df, state_unemp_df, feature_cols,
    rate_shock_pct=0.0, hpi_shock_pct=0.0, unemp_shock_pp=0.0,
):
    """
    Build RSF feature matrix from loan data + macro environment.
    
    For the base case, uses last observed macro values.
    For scenarios, applies shocks to the relevant features.
    
    Parameters
    ----------
    rate_shock_pct : float
        Parallel rate shock in percentage points (e.g., +1.0 = +100bp).
    hpi_shock_pct : float
        HPI shock as percentage change (e.g., -20.0 = -20%).
    unemp_shock_pp : float
        Unemployment shock in percentage points (e.g., +3.0).
    """
    N = len(loans_df)
    F = len(feature_cols)
    X = np.zeros((N, F), dtype=np.float64)
    feat_idx = {name: i for i, name in enumerate(feature_cols)}
    
    # Current macro values (with shocks)
    curr_mortgage = macro_df['MORTGAGE30US'].iloc[-1] + rate_shock_pct
    curr_dgs10 = macro_df['DGS10'].iloc[-1] + rate_shock_pct
    curr_dgs3mo = macro_df['DGS3MO'].iloc[-1] + rate_shock_pct
    
    # 12-month-ago values (historical, unshocked)
    m12 = min(12, len(macro_df) - 1)
    mortgage_12m = macro_df['MORTGAGE30US'].iloc[-m12]
    dgs10_12m = macro_df['DGS10'].iloc[-m12]
    dgs3mo_12m = macro_df['DGS3MO'].iloc[-m12]
    
    # Loan-level arrays
    int_rate = loans_df['int_rate'].values
    orig_upb = loans_df['orig_upb'].values
    fico = loans_df['fico_score'].values
    dti = loans_df['dti_r'].values
    ltv = loans_df['ltv_r'].values
    states = loans_df['property_state'].values
    orig_mortgage = loans_df['orig_MORTGAGE30US'].values
    orig_dgs10 = loans_df['orig_DGS10'].values
    orig_state_hpi = loans_df['orig_state_hpi'].values
    
    # Compute bal_repaid from amortization schedule
    term = loans_df['orig_loan_term'].values.astype(float)
    current_age = loans_df['current_loan_age'].values.astype(float)
    monthly_rate = int_rate / 100.0 / 12.0
    payment = np.where(
        monthly_rate > 0,
        orig_upb * monthly_rate / (1.0 - (1.0 + monthly_rate) ** (-term)),
        orig_upb / term,
    )
    factor = (1.0 + monthly_rate) ** current_age
    upb_now = np.where(
        monthly_rate > 0,
        orig_upb * factor - payment * (factor - 1.0) / monthly_rate,
        orig_upb - payment * current_age,
    )
    upb_now = np.maximum(upb_now, 0.0)
    bal_repaid = (orig_upb - upb_now) / orig_upb * 100.0
    
    # State-level HPI and unemployment (with shocks)
    hpi_factor = 1.0 + hpi_shock_pct / 100.0
    state_hpi_current = np.zeros(N)
    state_hpi_12m = np.zeros(N)
    state_unemp_current = np.zeros(N)
    state_unemp_12m = np.zeros(N)
    state_unemp_3m = np.zeros(N)
    national_hpi_current = 0.0
    
    hpi_cols = [c for c in state_hpi_df.columns if c.endswith('_hpi')]
    national_hpi_current = state_hpi_df[hpi_cols].iloc[-1].mean() * hpi_factor
    
    for i in range(N):
        st = states[i]
        hpi_col = f'{st}_hpi'
        unemp_col = f'{st}_unemployment'
        
        if hpi_col in state_hpi_df.columns:
            vals = state_hpi_df[hpi_col].dropna()
            state_hpi_current[i] = vals.iloc[-1] * hpi_factor
            state_hpi_12m[i] = vals.iloc[-min(12, len(vals))] if len(vals) >= 2 else vals.iloc[-1]
        
        if unemp_col in state_unemp_df.columns:
            vals = state_unemp_df[unemp_col].dropna()
            state_unemp_current[i] = vals.iloc[-1] + unemp_shock_pp
            state_unemp_12m[i] = vals.iloc[-min(12, len(vals))] if len(vals) >= 2 else vals.iloc[-1]
            state_unemp_3m[i] = vals.iloc[-min(3, len(vals))] if len(vals) >= 2 else vals.iloc[-1]
    
    # === Fill features ===
    # Static
    if 'int_rate' in feat_idx:
        X[:, feat_idx['int_rate']] = int_rate
    if 'log_upb' in feat_idx:
        X[:, feat_idx['log_upb']] = np.log(orig_upb)
    if 'fico_score' in feat_idx:
        X[:, feat_idx['fico_score']] = fico
    if 'dti_r' in feat_idx:
        X[:, feat_idx['dti_r']] = dti
    if 'ltv_r' in feat_idx:
        X[:, feat_idx['ltv_r']] = ltv
    
    # Behavioral
    if 'bal_repaid_lag1' in feat_idx:
        X[:, feat_idx['bal_repaid_lag1']] = bal_repaid
    if 't_act_12m' in feat_idx:
        X[:, feat_idx['t_act_12m']] = np.minimum(12.0, current_age)
    if 't_del_30d_12m' in feat_idx:
        X[:, feat_idx['t_del_30d_12m']] = 0.0  # assume performing
    if 't_del_60d_12m' in feat_idx:
        X[:, feat_idx['t_del_60d_12m']] = 0.0  # assume performing
    
    # Macro features
    if 'ppi_c_FRMA' in feat_idx:
        X[:, feat_idx['ppi_c_FRMA']] = int_rate - curr_mortgage
    if 'ppi_o_FRMA' in feat_idx:
        X[:, feat_idx['ppi_o_FRMA']] = int_rate - orig_mortgage
    if 'hpi_st_d_t_o' in feat_idx:
        X[:, feat_idx['hpi_st_d_t_o']] = state_hpi_current - orig_state_hpi
    if 'TB10Y_d_t_o' in feat_idx:
        X[:, feat_idx['TB10Y_d_t_o']] = curr_dgs10 - orig_dgs10
    if 'FRMA30Y_d_t_o' in feat_idx:
        X[:, feat_idx['FRMA30Y_d_t_o']] = curr_mortgage - orig_mortgage
    if 'hpi_st_log12m' in feat_idx:
        ratio = np.where(state_hpi_12m > 0, state_hpi_current / state_hpi_12m, 1.0)
        X[:, feat_idx['hpi_st_log12m']] = np.log(np.maximum(ratio, 1e-6))
    if 'hpi_r_st_us' in feat_idx:
        X[:, feat_idx['hpi_r_st_us']] = np.where(
            national_hpi_current > 0, state_hpi_current / national_hpi_current, 1.0
        )
    if 'st_unemp_r12m' in feat_idx:
        ratio = np.where(state_unemp_12m > 0, state_unemp_current / state_unemp_12m, 1.0)
        X[:, feat_idx['st_unemp_r12m']] = np.log(np.maximum(ratio, 1e-6))
    if 'st_unemp_r3m' in feat_idx:
        ratio = np.where(state_unemp_3m > 0, state_unemp_current / state_unemp_3m, 1.0)
        X[:, feat_idx['st_unemp_r3m']] = np.log(np.maximum(ratio, 1e-6))
    if 'TB10Y_r12m' in feat_idx:
        ratio = curr_dgs10 / dgs10_12m if dgs10_12m > 0 else 1.0
        X[:, feat_idx['TB10Y_r12m']] = np.log(max(ratio, 1e-6))
    if 'T10Y3MM' in feat_idx:
        X[:, feat_idx['T10Y3MM']] = curr_dgs10 - curr_dgs3mo
    if 'T10Y3MM_r12m' in feat_idx:
        spread_now = curr_dgs10 - curr_dgs3mo
        spread_12m = dgs10_12m - dgs3mo_12m
        X[:, feat_idx['T10Y3MM_r12m']] = (
            (spread_now - spread_12m) / abs(spread_12m) if abs(spread_12m) > 1e-6 else 0.0
        )
    
    return X

print('build_rsf_features() defined.')

In [ ]:
%%time
# Build base feature matrix and project cash flows
print('Building base feature matrix...')
X_base = build_rsf_features(
    portfolio, macro_df, state_hpi_df, state_unemp_df, FEATURE_COLS,
)
print(f'Feature matrix: {X_base.shape}')

print('Projecting cash flows...')
cf_base = engine.project_cash_flows(portfolio, X_base)

# Aggregate to portfolio level
portfolio_cf = engine.aggregate_portfolio(cf_base)

print(f'\n=== Portfolio Cash Flow Summary (RSF, Base Scenario) ===')
print(f'Total expected CF: ${portfolio_cf["total_cf"].sum():,.0f}')
print(f'Total interest: ${portfolio_cf["interest"].sum():,.0f}')
print(f'Total sched principal: ${portfolio_cf["scheduled_principal"].sum():,.0f}')
print(f'Total prepayment: ${portfolio_cf["prepayment"].sum():,.0f}')
print(f'Total recovery: ${portfolio_cf["recovery"].sum():,.0f}')
print(f'Total loss: ${portfolio_cf["loss"].sum():,.0f}')
print(f'\nAvg survival at end: {portfolio_cf["avg_survival"].iloc[-1]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

months = portfolio_cf['month'].values

# Stacked cash flows
ax = axes[0, 0]
ax.stackplot(months,
    portfolio_cf['interest'] / 1e6,
    portfolio_cf['scheduled_principal'] / 1e6,
    portfolio_cf['prepayment'] / 1e6,
    portfolio_cf['recovery'] / 1e6,
    labels=['Interest', 'Sched Principal', 'Prepayment', 'Recovery'],
    colors=['#4e79a7', '#59a14f', '#f28e2b', '#e15759'],
    alpha=0.8)
ax.set_title('Portfolio Monthly Cash Flows (RSF)')
ax.set_xlabel('Month')
ax.set_ylabel('$M')
ax.legend(loc='upper right', fontsize=8)

# Cumulative cash flows
ax = axes[0, 1]
ax.plot(months, portfolio_cf['cumulative_cf'] / 1e6, 'k-', lw=2)
ax.set_title('Cumulative Portfolio Cash Flows')
ax.set_xlabel('Month')
ax.set_ylabel('$M (cumulative)')

# Average survival
ax = axes[1, 0]
ax.plot(months, portfolio_cf['avg_survival'], 'k-', lw=2)
ax.set_title('Average Portfolio Survival Rate')
ax.set_xlabel('Month')
ax.set_ylabel('S(t)')

# Loss over time
ax = axes[1, 1]
ax.plot(months, portfolio_cf['loss'].cumsum() / 1e6, 'r-', lw=2)
ax.set_title('Cumulative Expected Loss')
ax.set_xlabel('Month')
ax.set_ylabel('$M (cumulative)')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle('Portfolio Cash Flow Projection - RSF Base Scenario', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_rsf_portfolio_base_scenario.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compute risk metrics for base scenario
discount_rate = macro_df['MORTGAGE30US'].iloc[-1] / 100.0

base_metrics = compute_all_risk_metrics(portfolio_cf, annual_rate=discount_rate)

print(f'=== Base Scenario Risk Metrics (RSF) ===')
print(f'Discount rate: {discount_rate*100:.2f}%')
print(f'NPV: ${base_metrics["npv"]:,.0f}')
print(f'Modified Duration: {base_metrics["modified_duration"]:.2f} years')
print(f'Modified Convexity: {base_metrics["modified_convexity"]:.2f}')
print(f'WAL: {base_metrics["wal_years"]:.2f} years')
print(f'Total Cash Flow: ${base_metrics["total_cash_flow"]:,.0f}')
print(f'Total Loss: ${base_metrics["total_loss"]:,.0f}')

---

## 6. Scenario Analysis

Since RSF uses snapshot features, scenario analysis modifies the feature vector
directly (shifting rate-sensitive and macro features) and re-predicts survival
curves. This captures non-linear interactions that the Cox model's proportional
hazards assumption cannot.

In [ ]:
# Define scenarios as (rate_shock_pct, hpi_shock_pct, unemp_shock_pp)
SCENARIOS = {
    'base': (0.0, 0.0, 0.0),
    'rate_+100bp': (1.0, 0.0, 0.0),
    'rate_+200bp': (2.0, 0.0, 0.0),
    'rate_-100bp': (-1.0, 0.0, 0.0),
    'rate_-200bp': (-2.0, 0.0, 0.0),
    'hpi_stress': (0.0, -20.0, 0.0),
    'recession': (-1.5, -15.0, 3.0),
    'severe_recession': (-2.0, -30.0, 6.0),
}

print(f'Defined {len(SCENARIOS)} scenarios:')
for name, (rs, hs, us) in SCENARIOS.items():
    print(f'  {name}: rate_shock={rs:+.1f}%, hpi_shock={hs:+.1f}%, unemp_shock={us:+.1f}pp')

In [ ]:
%%time
# Run all scenarios
scenario_results = {}

for name, (rate_shock, hpi_shock, unemp_shock) in SCENARIOS.items():
    print(f'Running scenario: {name}...')
    
    # Build features with shocks
    X_scen = build_rsf_features(
        portfolio, macro_df, state_hpi_df, state_unemp_df, FEATURE_COLS,
        rate_shock_pct=rate_shock,
        hpi_shock_pct=hpi_shock,
        unemp_shock_pp=unemp_shock,
    )
    
    # Project cash flows
    cf_scen = engine.project_cash_flows(portfolio, X_scen)
    agg_scen = engine.aggregate_portfolio(cf_scen)
    
    # Determine discount rate for this scenario
    if 'rate_+' in name:
        disc_rate = discount_rate + rate_shock / 100.0
    elif 'rate_-' in name:
        disc_rate = discount_rate + rate_shock / 100.0
    elif name in ('recession', 'severe_recession'):
        disc_rate = discount_rate + rate_shock / 100.0
    else:
        disc_rate = discount_rate
    
    metrics = compute_all_risk_metrics(agg_scen, annual_rate=disc_rate)
    
    scenario_results[name] = {
        'portfolio_cf': agg_scen,
        'metrics': metrics,
        'discount_rate': disc_rate,
    }
    
    print(f'  NPV: ${metrics["npv"]:,.0f}, Duration: {metrics["modified_duration"]:.2f}y, WAL: {metrics["wal_years"]:.2f}y')

print('\nAll scenarios complete.')

---

## 7. Risk Metrics Comparison

In [ ]:
# Create comparison table
metrics_rows = []
for name, res in scenario_results.items():
    m = res['metrics']
    metrics_rows.append({
        'Scenario': name,
        'Discount Rate': f"{res['discount_rate']*100:.2f}%",
        'NPV ($M)': m['npv'] / 1e6,
        'Mod Duration (yr)': m['modified_duration'],
        'Mod Convexity': m['modified_convexity'],
        'WAL (yr)': m['wal_years'],
        'Total CF ($M)': m['total_cash_flow'] / 1e6,
        'Total Loss ($M)': m['total_loss'] / 1e6,
    })

metrics_df = pd.DataFrame(metrics_rows).set_index('Scenario')
print('=== Risk Metrics Comparison (RSF) ===')
print(metrics_df.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scenario_names = list(scenario_results.keys())
x = np.arange(len(scenario_names))

# Color scheme
colors = [
    '#4e79a7' if s == 'base'
    else '#59a14f' if 'rate_-' in s
    else '#e15759' if 'rate_+' in s
    else '#f28e2b'
    for s in scenario_names
]

# NPV
ax = axes[0, 0]
npvs = [scenario_results[s]['metrics']['npv'] / 1e6 for s in scenario_names]
ax.bar(x, npvs, color=colors, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(scenario_names, rotation=45, ha='right', fontsize=8)
ax.set_title('NPV by Scenario')
ax.set_ylabel('$M')

# Duration
ax = axes[0, 1]
durations = [scenario_results[s]['metrics']['modified_duration'] for s in scenario_names]
ax.bar(x, durations, color=colors, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(scenario_names, rotation=45, ha='right', fontsize=8)
ax.set_title('Modified Duration by Scenario')
ax.set_ylabel('Years')

# WAL
ax = axes[1, 0]
wals = [scenario_results[s]['metrics']['wal_years'] for s in scenario_names]
ax.bar(x, wals, color=colors, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(scenario_names, rotation=45, ha='right', fontsize=8)
ax.set_title('Weighted Average Life by Scenario')
ax.set_ylabel('Years')

# Total Loss
ax = axes[1, 1]
losses = [scenario_results[s]['metrics']['total_loss'] / 1e6 for s in scenario_names]
ax.bar(x, losses, color=colors, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(scenario_names, rotation=45, ha='right', fontsize=8)
ax.set_title('Total Expected Loss by Scenario')
ax.set_ylabel('$M')

for ax in axes.flat:
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Risk Metrics Across Scenarios (RSF)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_rsf_scenario_risk_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Overlay portfolio cash flows across key scenarios
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

key_scenarios = ['base', 'rate_+200bp', 'rate_-200bp', 'recession', 'severe_recession']
scenario_colors = {
    'base': 'black', 'rate_+200bp': '#e15759', 'rate_-200bp': '#59a14f',
    'recession': '#f28e2b', 'severe_recession': '#b07aa1',
}

for name in key_scenarios:
    if name in scenario_results:
        cf = scenario_results[name]['portfolio_cf']
        lw = 2.5 if name == 'base' else 1.5
        ls = '-' if name == 'base' else '--'
        
        axes[0].plot(cf['month'], cf['total_cf'] / 1e6,
                     color=scenario_colors.get(name, 'gray'), lw=lw, ls=ls, label=name)
        axes[1].plot(cf['month'], cf['avg_survival'],
                     color=scenario_colors.get(name, 'gray'), lw=lw, ls=ls, label=name)

axes[0].set_title('Monthly Total Cash Flow')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('$M')
axes[0].legend(fontsize=8)

axes[1].set_title('Average Survival')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('S(t)')
axes[1].legend(fontsize=8)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_rsf_scenario_cf_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. Portfolio Segmentation

In [ ]:
# Compute loan-level NPV and WAL under base scenario
loan_npv = compute_npv(cf_base['total_cf'], annual_rate=discount_rate)
loan_principal_return = cf_base['scheduled_principal'] + cf_base['prepayment'] + cf_base['recovery']

# Per-loan WAL
T_base = cf_base['total_cf'].shape[1]
t_arr = np.arange(1, T_base + 1)
loan_total_prin = loan_principal_return.sum(axis=1)
loan_wal = np.where(
    loan_total_prin > 1e-6,
    np.sum(t_arr[None, :] * loan_principal_return, axis=1) / loan_total_prin / 12.0,
    0.0
)

# Add to portfolio df
portfolio = portfolio.copy()
portfolio['npv'] = loan_npv
portfolio['wal_years'] = loan_wal

# Create bins
portfolio['fico_band'] = pd.cut(portfolio['fico_score'], bins=[0, 680, 720, 760, 800, 900],
                                 labels=['<680', '680-720', '720-760', '760-800', '800+'])
portfolio['ltv_band'] = pd.cut(portfolio['ltv_r'], bins=[0, 60, 70, 80, 90, 100],
                                labels=['<60', '60-70', '70-80', '80-90', '90+'])

print(f'Loan-level metrics computed for {len(portfolio):,} loans')

In [ ]:
# By vintage
vintage_metrics = portfolio.groupby('vintage_year').agg(
    n_loans=('loan_sequence_number', 'count'),
    avg_rate=('int_rate', 'mean'),
    avg_fico=('fico_score', 'mean'),
    total_upb=('orig_upb', 'sum'),
    total_npv=('npv', 'sum'),
    avg_wal=('wal_years', 'mean'),
).round(2)
vintage_metrics['total_upb_M'] = (vintage_metrics['total_upb'] / 1e6).round(1)
vintage_metrics['total_npv_M'] = (vintage_metrics['total_npv'] / 1e6).round(1)

print('=== Risk Metrics by Vintage (RSF) ===')
print(vintage_metrics[['n_loans', 'avg_rate', 'avg_fico', 'total_upb_M', 'total_npv_M', 'avg_wal']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By FICO
fico_metrics = portfolio.groupby('fico_band', observed=True).agg(
    n_loans=('loan_sequence_number', 'count'),
    avg_npv=('npv', 'mean'),
    avg_wal=('wal_years', 'mean'),
).reset_index()

ax = axes[0]
x = np.arange(len(fico_metrics))
width = 0.35
ax2 = ax.twinx()
ax.bar(x - width/2, fico_metrics['avg_npv'] / 1e3, width, color='steelblue', alpha=0.8, label='Avg NPV ($K)')
ax2.plot(x, fico_metrics['avg_wal'], 'o-', color='indianred', lw=2, label='Avg WAL (yr)')
ax.set_xticks(x)
ax.set_xticklabels(fico_metrics['fico_band'])
ax.set_xlabel('FICO Band')
ax.set_ylabel('Avg NPV ($K)', color='steelblue')
ax2.set_ylabel('Avg WAL (yr)', color='indianred')
ax.set_title('Risk Metrics by FICO (RSF)')
ax.grid(True, alpha=0.3, axis='y')

# By LTV
ltv_metrics = portfolio.groupby('ltv_band', observed=True).agg(
    n_loans=('loan_sequence_number', 'count'),
    avg_npv=('npv', 'mean'),
    avg_wal=('wal_years', 'mean'),
).reset_index()

ax = axes[1]
x = np.arange(len(ltv_metrics))
ax2 = ax.twinx()
ax.bar(x - width/2, ltv_metrics['avg_npv'] / 1e3, width, color='steelblue', alpha=0.8, label='Avg NPV ($K)')
ax2.plot(x, ltv_metrics['avg_wal'], 'o-', color='indianred', lw=2, label='Avg WAL (yr)')
ax.set_xticks(x)
ax.set_xticklabels(ltv_metrics['ltv_band'])
ax.set_xlabel('LTV Band')
ax.set_ylabel('Avg NPV ($K)', color='steelblue')
ax2.set_ylabel('Avg WAL (yr)', color='indianred')
ax.set_title('Risk Metrics by LTV (RSF)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alm_rsf_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compute effective duration and convexity (with macro covariate shocks)
print('Computing effective duration (100bp shock)...')

shock_bps = 100
dr = shock_bps / 10000.0

# Base
npv_base = compute_npv(portfolio_cf['total_cf'].values, discount_rate)

# Rate up: rebuild features with +100bp shock
X_up = build_rsf_features(
    portfolio, macro_df, state_hpi_df, state_unemp_df, FEATURE_COLS,
    rate_shock_pct=1.0,
)
cf_up = engine.project_cash_flows(portfolio, X_up)
npv_up = compute_npv(cf_up['total_cf'].sum(axis=0), discount_rate + dr)

# Rate down: rebuild features with -100bp shock
X_down = build_rsf_features(
    portfolio, macro_df, state_hpi_df, state_unemp_df, FEATURE_COLS,
    rate_shock_pct=-1.0,
)
cf_down = engine.project_cash_flows(portfolio, X_down)
npv_down = compute_npv(cf_down['total_cf'].sum(axis=0), discount_rate - dr)

eff_dur = -(npv_up - npv_down) / (2.0 * dr * npv_base)
eff_conv = (npv_up + npv_down - 2.0 * npv_base) / (dr ** 2 * npv_base)

print(f'\n=== Duration/Convexity Comparison (RSF) ===')
print(f'Modified Duration:  {base_metrics["modified_duration"]:.2f} years')
print(f'Effective Duration: {eff_dur:.2f} years')
print(f'Modified Convexity:  {base_metrics["modified_convexity"]:.2f}')
print(f'Effective Convexity: {eff_conv:.2f}')
print(f'\nEffective < Modified duration indicates negative convexity from prepayment optionality')

---

## Summary

### Key Results
- Converted cause-specific RSF models into projected mortgage cash flows
- RSF hazards extracted from survival curves: h_k(t) = 1 - S_k(t)/S_k(t-1)
- **Backtested on held-out fold 10**: compared predicted vs realized CIF and survival
- Validated single-loan accounting identity: S(T) + CIF_prepay(T) + CIF_default(T) = 1
- Projected portfolio-level cash flows under 8 scenarios (base + 7 stress)
- Computed interest rate risk metrics (NPV, duration, convexity, WAL)
- Analyzed risk metrics by vintage, FICO, and LTV segments

### RSF vs Cox Comparison
- RSF captures non-linear effects and feature interactions without proportional hazards assumption
- RSF uses snapshot features (not time-varying), so scenario analysis modifies feature vectors directly
- RSF hazards are individual-specific (heterogeneity across loans even with same features)
- Trade-off: RSF is more flexible but less interpretable than Cox

### Sanity Checks
- S(T) + CIF_prepay(T) + CIF_default(T) = 1 (accounting identity)
- Predicted CIF tracks realized CIF on out-of-sample fold 10
- Prepayments accelerate when rates fall, slow when rates rise
- Effective duration < Modified duration (prepayment optionality)
- Default losses increase under HPI stress and recession scenarios